# 05 — Similarity facts

Turn each DayHike in one region into a row of similarity facts, one Facet at a
time, and write that table next to the Catalog. `05a_similarity.ipynb` compares
DayHikes from it. This is the notebook design phase of ADR-0011 (proposed):
production Ranking, the CLI and the Catalog do not change.

**Region:** DayHikes whose Trailhead lies in lat 42.3–46.9, lon 13.3–19.5
(Slovenia + Croatia: Julian Alps, karst, Velebit, the coast, the Pannonian
lowland). `load_region` reads the GeometryStore the way `04a_geometry.ipynb`
does (vertices, amenities, dayhikes, ways as GeoDataFrames) with the Catalog's
`feature_vectors`, Trailhead and Difficulty merged onto `dayhikes`. Rows are
filtered by region DayHike id in SQL; the full GeometryStore is ~21 GB.

Open `data/catalog.sqlite` plus `data/catalog.geometry.sqlite` when both exist.
Otherwise (and always under pytest) build the committed OSM fixture
`tests/fixtures/catalog.osm`, whose DayHikes all lie inside the same box.

**Facets here:** Effort, Altitude, Profile, Landscape, Weather, Water, Facilities, Trail,
RoundTrip, Steepness.

A missing fact stays missing (NaN), never 0: a DayHike without elevation is
not a flat DayHike.

In [ ]:
import json
import os
import tempfile
from collections import Counter
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point
from shapely.ops import substring

from dayhike import build_catalog, geometry_store_path_for
from dayhike.catalog import GAP_JOIN_M
from dayhike.similarity import (
    CLIMATE_MONTH_COLUMNS,
    CORRIDOR_M,
    FACILITIES_FLAG_COLUMNS,
    LANDCOVER_SHARE_COLUMNS,
    SURFACE_SHARE_COLUMNS,
    STEEPEST_KM_WINDOW_M,
    STEEPNESS_STEP_M,
    TRAILHEAD_M,
    WATER_SOURCE_KINDS,
    altitude_facts,
    corridor_amenities,
    effort_facts,
    facilities_facts,
    landscape_facts,
    load_region,
    profile_facts,
    profile_lockstep,
    round_trip_facts,
    similarity_facts_path_for,
    steepness_facts,
    surface_class,
    surface_class_from_tag,
    trail_facts,
    water_facts,
    weather_facts,
)

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")
import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)


repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

local = repo / "data" / "catalog.sqlite"
if (
    local.is_file()
    and geometry_store_path_for(local).is_file()
    and "PYTEST_CURRENT_TEST" not in os.environ
):
    catalog_path = local
else:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    build_catalog(fixture, catalog_path)

REGION_LAT = (42.3, 46.9)
REGION_LON = (13.3, 19.5)
vertices, amenities, dayhikes, ways = load_region(
    catalog_path, lat_range=REGION_LAT, lon_range=REGION_LON
)
print("catalog", catalog_path)
for name, frame in (
    ("vertices", vertices),
    ("amenities", amenities),
    ("dayhikes", dayhikes),
    ("ways", ways),
):
    print(f"region {name}\t{len(frame)}")
print("name_origin", dayhikes["name_origin"].value_counts().to_dict())

## Effort

How much work the day is. Three facts per DayHike:

- `length_m`: the Catalog length of the main walk.
- `elevation_gain_m`: metres climbed along the walk. OSM way order is
  arbitrary, so a point-to-point DayHike stored "downhill" would show almost no
  climb. The fact is the larger of the climb and the descent along the walk,
  which is the same in both directions. On a loop the two are equal. Vertices
  without `ele` are skipped (the Catalog does the same); no `ele` at all leaves
  the gain missing.
- `difficulty_rank`: Difficulty as the SAC rank 1–6 (`hiking` 1 …
  `difficult_alpine_hiking` 6), from the strictest `sac_scale` on member ways.
  Missing when no member way is tagged, which is common.

In [ ]:
effort = effort_facts(dayhikes, vertices)
print(effort.describe().round(1).to_string())
print("effort missing share")
print(effort.isna().mean().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(effort["length_m"].dropna() / 1000, bins=40, color="#0f766e")
axes[0].set_xlabel("length (km)")
axes[1].hist(effort["elevation_gain_m"].dropna(), bins=40, color="#c2410c")
axes[1].set_xlabel("elevation gain (m)")
ranks = effort["difficulty_rank"].value_counts().sort_index()
axes[2].bar(ranks.index.astype(int).astype(str), ranks.to_numpy(), color="#7c3aed")
axes[2].set_xlabel("Difficulty rank (SAC)")
for ax in axes:
    ax.set_ylabel("DayHikes")
fig.suptitle("Effort facts in the region")
fig.tight_layout()
show_fig(fig)

## Altitude

How high the DayHike goes, apart from how much it climbs. Two facts, read from
the Catalog's `feature_vectors` (FABDEM when `sample-dem` has run, else OSM
`ele`):

- `highest_m`: the highest vertex.
- `lowest_m`: the lowest vertex. A coastal climb from sea level and a walk from
  a 1500 m pass can reach the same top.

Both are the same whichever way the DayHike is walked.

In [ ]:
altitude = altitude_facts(dayhikes)
print(altitude.describe().round(1).to_string())
print("altitude missing share")
print(altitude.isna().mean().round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = 40
ax.hist(altitude["lowest_m"].dropna(), bins=bins, alpha=0.6, color="#1d4ed8", label="lowest")
ax.hist(altitude["highest_m"].dropna(), bins=bins, alpha=0.6, color="#c2410c", label="highest")
ax.set_xlabel("elevation (m)")
ax.set_ylabel("DayHikes")
ax.set_title("Altitude facts in the region")
ax.legend()
fig.tight_layout()
show_fig(fig)

## Profile

The form of the elevation plot, apart from how high the DayHike goes and how
much it climbs. One fact per DayHike, `profile_shape`, its ProfileShape:

1. Cumulative distance along the vertices in stored order: haversine
   metres between consecutive vertices, over every vertex.
2. Elevation at 64 equal steps of the distance fraction 0 → 1, interpolated
   between the vertices that have `ele` (the ElevationProfile, resampled).
3. Subtract the lowest of the 64 values and divide by max(range, 50 m), so
   height is in [0, 1]. A DayHike that varies by less than 50 m is not
   stretched: 5 m of noise stays at most 0.1, and a flat walk looks flat.

Fewer than two vertices with `ele` leaves the ProfileShape missing. The shape
keeps the stored direction; `05a` compares it forward and reversed, so OSM's
way order does not matter.

The figure picks, for four contrasting forms, the region DayHike with an OSM
name whose ProfileShape is closest (`profile_lockstep`) to a drawn template: a single
summit (up and down), a traverse (one climb end to end), rolling (several
climbs) and flat. Each DayHike is drawn in its stored direction, so a traverse
may run downhill.

In [ ]:
profile = profile_facts(dayhikes, vertices)
shapes = profile["profile_shape"].dropna()
print(f"profile missing share\t{1 - len(shapes) / len(profile):.3f}")
print(f"profile range under 50 m share\t{shapes.map(lambda s: s.max() < 1).mean():.3f}")

fraction = np.linspace(0.0, 1.0, 64)
templates = {
    "single summit": np.interp(fraction, [0.0, 0.5, 1.0], [0.0, 1.0, 0.0]),
    "traverse": fraction,
    "rolling": (1 - np.cos(fraction * 8 * np.pi)) / 2,
    "flat": np.zeros(64),
}
hikes = dayhikes.set_index("id")
names = hikes["name"]
named = shapes[hikes.loc[shapes.index, "name_origin"] == "osm_name"]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, (form, template) in zip(axes, templates.items()):
    closest = named.map(lambda s: profile_lockstep(template, s)).idxmax()
    print(f"profile_example\t{form}\t{names[closest]}")
    ax.plot(fraction, template, color="#a8a29e", linestyle="--", label="template")
    ax.plot(fraction, shapes[closest], color="#0f766e", label="DayHike")
    ax.set_title(f"{form}\n{names[closest]}", fontsize=8)
    ax.set_xlabel("fraction of length")
axes[0].set_ylabel("ProfileShape height")
axes[0].legend(fontsize=8)
fig.tight_layout()
show_fig(fig)

## Landscape

What the DayHike looks like, apart from how hard or how high it is. Nine facts
per DayHike:

- Six LandCover shares (`tree_share`, `shrub_share`, `grass_share`,
  `bare_share`, `snow_share`, `other_share`). The first five are the Catalog's
  `feature_vectors` fractions (each GeometryStore vertex counts once). The
  sixth is the leftover, 1 − their sum, clipped at 0: WorldCover classes the
  Catalog does not keep (cropland, built-up, wetland, water) land there, so
  the six shares always sum to 1 and can be compared as a mix. When the
  Catalog fractions are missing (a hike outside the sampled rasters) all six
  are missing, not 0.
- `peak_summited`: a Peak (`natural=peak` amenity) within 50 m of the line.
- `viewpoint_along`: a Viewpoint (`tourism=viewpoint`) inside the 100 m
  Corridor.
- `peaks_around`: the number of Peaks within 2 km of the line.

**Corridor.** An amenity point is along a DayHike when it is within 100 m of
DayHikeGeometry, measured to the line and not only to the vertices (vertex
spacing reaches 120 m, and a way can have one segment of several hundred
metres). `corridor_amenities(vertices, amenities, kind=..., radius_m=...)`
inserts points along each segment, at most radius / 10 apart, and asks a
haversine BallTree for the amenity points near them. It returns one row per
(DayHike, point) with the distance to the line, so any amenity kind and any
radius work. The Water and Facilities Facets reuse it. All three radii here
(50 m, 100 m, 2 km) use the same helper, so the line is read the same way
whichever way the DayHike is walked.

Peaks and Viewpoints are raw GeometryStore amenities, not Catalog
AmenityFacts: they are looked up around the line, not the Trailhead.

In [ ]:
landscape = landscape_facts(dayhikes, vertices, amenities)
print(landscape.describe().round(3).to_string())
print("landscape missing share")
print(landscape.isna().mean().round(3).to_string())
print(f"peak_summited_hikes\t{int(landscape['peak_summited'].sum())}\tof\t{len(landscape)}")
print(f"viewpoint_along_hikes\t{int(landscape['viewpoint_along'].sum())}\tof\t{len(landscape)}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
shares = landscape[list(LANDCOVER_SHARE_COLUMNS)].dropna()
colors = ["#15803d", "#84cc16", "#facc15", "#a8a29e", "#38bdf8", "#e7e5e4"]
left = 0.0
for column, color in zip(LANDCOVER_SHARE_COLUMNS, colors):
    width = shares[column].mean()
    axes[0].barh([0], [width], left=left, color=color, edgecolor="white", label=column[:-6])
    left += width
axes[0].set_xlim(0, 1)
axes[0].set_yticks([])
axes[0].set_xlabel("mean share of the DayHike")
axes[0].set_title("LandCover mix (six shares)", fontsize=10)
axes[0].legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.3))
axes[1].hist(shares["tree_share"], bins=20, color="#15803d")
axes[1].set_xlabel("tree share")
axes[1].set_ylabel("DayHikes")
axes[2].hist(landscape["peaks_around"], bins=30, color="#7c3aed")
axes[2].set_xlabel("Peaks within 2 km")
axes[2].set_ylabel("DayHikes")
fig.suptitle("Landscape facts in the region")
fig.tight_layout()
show_fig(fig)

## Weather

What weather the DayHike is likely to have, and how much sun its slopes get.
Forty facts per DayHike:

- The ClimateNormal regime, read from the Catalog's `feature_vectors`
  (WorldClim 2.1, 1970–2000, averaged along DayHikeGeometry):
  `temperature_c` (June–August mean, °C), `precipitation_mm` (mean monthly
  precipitation over the year, mm) and `solar_radiation` (June–August mean,
  kJ m⁻² day⁻¹). This is the slice used when the Hiker names no QueryMonth.
- The twelve months, `temperature_c_01` … `solar_radiation_12`, read from the
  Catalog's `climate_months`. `05a` has one `QUERY_MONTH` setting that swaps
  the regime for one month's three values (ADR-0010).
- `southness`: SolarAspect, how south-facing the slopes along the DayHike are.
  It is terrain read from the DEM, not weather (shade from trees is
  LandCover). It sits in this Facet because it says how much sun the hike
  gets.

**SolarAspect rule** (`solar_aspect_southness`, FABDEM 1° tiles in `data/dem/`,
ADR-0008):

1. About 32 vertices per DayHike: every k-th vertex counted from the nearer
   end of the walk (k = ⌈vertices / 32⌉), so the same vertices are read whichever
   way the DayHike is walked.
2. FABDEM is read at the vertex and 2 arc-seconds east and north of it. That
   is two FABDEM cells (about 62 m north, 44 m east at 45° N), so the offset
   always reads a different cell, at a known distance.
3. The rise to the east and to the north give the slope and the downslope
   direction. Southness = (1 − the north part of the downslope direction) / 2:
   1 due south, 0 due north, 0.5 east or west, 0.85 south-east or south-west.
4. Points with a slope under 5° are left out. A valley floor or a broad
   ridge top faces no way, and a metre or two of FABDEM noise over 44–62 m
   would give it a random direction. The DayHike's southness is the mean over
   its sloped points. With none (a flat walk) or with no covering FABDEM tile
   (the committed fixture has none), southness is missing, not 0.5.

Notebook `04`'s prototype scored (1 + cos(aspect − π)) / 2 with the aspect
measured from east, which gives 1 for a west-facing slope and 0.5 for a
south-facing one. It is not used here. The rule above is tested on synthetic
north-, south-, east- and west-facing ramp rasters in `tests/test_similarity.py`.

On a mostly flat walk the few sloped points (an embankment, a river bank)
decide southness, so its value says little; it is still a value, because
some slope was found. The maps therefore show the three most north-facing and
the three most south-facing named DayHikes of at least 3 km with at least
500 m between their lowest and highest point, on a topographic basemap (the
line alone when offline).

In [ ]:
dem_dir = catalog_path.parent / "dem"
weather = weather_facts(dayhikes, vertices, dem_dir)
WEATHER_REGIME = ["temperature_c", "precipitation_mm", "solar_radiation", "southness"]
print("dem", dem_dir)
print(weather[WEATHER_REGIME].describe().round(2).to_string())
print("weather missing share")
print(weather[WEATHER_REGIME].isna().mean().round(3).to_string())
print(f"climate_months_missing_share\t{weather[list(CLIMATE_MONTH_COLUMNS)].isna().any(axis=1).mean():.3f}")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
labels = [
    "June–August temperature (°C)",
    "mean monthly precipitation (mm)",
    "June–August solar radiation (kJ m⁻² day⁻¹)",
    "southness (0 north, 1 south)",
]
for ax, column, label, color in zip(axes, WEATHER_REGIME, labels, ["#c2410c", "#1d4ed8", "#ca8a04", "#7c3aed"]):
    ax.hist(weather[column].dropna(), bins=30, color=color)
    ax.set_xlabel(label, fontsize=8)
axes[0].set_ylabel("DayHikes")
fig.suptitle("Weather facts in the region (ClimateNormal regime and SolarAspect)")
fig.tight_layout()
show_fig(fig)

hikes = dayhikes.set_index("id")
facing = weather[["southness"]].join(altitude).join(hikes[["name", "name_origin", "length_m", "geometry"]])
facing = facing[
    (facing["name_origin"] == "osm_name")
    & (facing["length_m"] >= 3_000)
    & (facing["highest_m"] - facing["lowest_m"] >= 500)
].dropna(subset=["southness"])
if facing.empty:
    print("southness missing everywhere: no FABDEM tile covers these DayHikes")
else:
    examples = pd.concat([facing.nsmallest(3, "southness"), facing.nlargest(3, "southness")])
    fig, axes = plt.subplots(1, len(examples), figsize=(3 * len(examples), 3.2))
    for ax, (dayhike_id, row) in zip(np.atleast_1d(axes), examples.iterrows()):
        print(f"southness_example\t{row['name']}\t{row['southness']:.2f}")
        one = gpd.GeoDataFrame([row], geometry="geometry", crs=dayhikes.crs).to_crs(epsg=3857)
        one.plot(ax=ax, color="#1d4ed8" if row["southness"] < 0.5 else "#c2410c", linewidth=2)
        try:
            import contextily as cx

            cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
        except Exception as error:  # offline: the line alone is still the map
            print("basemap skipped", type(error).__name__)
        ax.set_title(f"{row['name']}\nsouthness {row['southness']:.2f}", fontsize=8)
        ax.set_axis_off()
    fig.suptitle("Most north-facing (blue) and most south-facing (orange) named DayHikes")
    fig.tight_layout()
    show_fig(fig)

## Water

How much water the DayHike passes. Two facts per DayHike:

- `water_count`: the number of WaterSources in the 100 m Corridor, the same band
  as Landscape's Viewpoint: GeometryStore amenities of kind `drinking_water`,
  `spring`, `water_tap` or `water_point` (`WATER_SOURCE_KINDS`).
  The fact is the real count; the 05a measure caps it at three, so "none", "one",
  "a couple" and "plenty" are the levels that matter.
- `dry_stretch_m`: DryStretch, the longest distance walked without passing one.

**DryStretch rule** (`water_facts`):

1. The Corridor points come from `corridor_amenities`, so a point beside the
   middle of a long segment counts, and the distance is to the line.
2. Each point is placed at its position along the DayHike: the place on the
   walk closest to it, in metres from the first vertex (haversine along every
   vertex). Where a path passes the same fountain twice, the closer pass
   counts.
3. The walk is cut at those positions. Stretches run from the first vertex to the
   first point, between points, and from the last point to the last vertex. The
   longest is the DryStretch. Two points at one position cut the walk once.
4. With no point in the Corridor the DryStretch is the length of the walk.
   A water point at the very start or end cuts nothing: a hike whose
   only fountain is at the car park still has a DryStretch of almost its whole
   length.

The positions are measured along the line, so reversing the walk turns every
position into length minus position and leaves the DryStretch alone.

**What counts as water.** Ingest keeps OSM `amenity=drinking_water`,
`natural=spring`, `man_made=water_tap` and `amenity=water_point`, and drops a
spring, tap or water point tagged `drinking_water=no` (CONTEXT.md WaterSource).
A spring or tap that is also tagged `amenity=drinking_water` stays a
drinking-water point, so the Catalog's drinking-water AmenityFact is the same
as before springs were kept. Streams, troughs and unmapped springs are still
not counted, so a backcountry hike can look drier than it is.

The code below prints the region's WaterSources by kind and the split by how
high the DayHike goes. The region box holds 10,325 drinking-water points,
5,759 springs, 617 taps and 190 water points. Springs moved the mountains most:
55% of the DayHikes that reach above 1,500 m pass no WaterSource, 62% of those
reaching 500 to 1,500 m and 55% of those under 500 m. When only drinking water
counted, on the Catalog before the main walk, those shares were 73%, 74% and
65%. Of the region, 58% of the DayHikes pass no WaterSource (71% with drinking
water alone before the main walk).

In [ ]:
water = water_facts(dayhikes, vertices, amenities)
print(water.describe().round(1).to_string())
print("water missing share")
print(water.isna().mean().round(3).to_string())
for kind in WATER_SOURCE_KINDS:
    print(f"water_source_in_region_box\t{kind}\t{int((amenities['kind'] == kind).sum())}")
print(f"water_zero_share\t{(water['water_count'] == 0).mean():.3f}")
print(f"water_three_or_more_share\t{(water['water_count'] >= 3).mean():.3f}")

bands = pd.cut(
    altitude["highest_m"], [-1, 500, 1500, 10_000], labels=["under 500 m", "500-1500 m", "over 1500 m"]
)
by_band = water.groupby(bands, observed=True).agg(
    hikes=("water_count", "size"),
    no_water_share=("water_count", lambda c: (c == 0).mean()),
    median_dry_stretch_km=("dry_stretch_m", lambda d: d.median() / 1000),
)
print("water by highest point")
print(by_band.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
levels = water["water_count"].clip(upper=6).value_counts().sort_index()
axes[0].bar(levels.index.astype(int).astype(str), levels.to_numpy(), color="#1d4ed8")
axes[0].set_xticks(range(len(levels)), [str(int(i)) if i < 6 else "6+" for i in levels.index])
axes[0].set_xlabel("WaterSources in the Corridor")
axes[0].set_ylabel("DayHikes")
axes[1].hist(water["dry_stretch_m"].dropna() / 1000, bins=40, color="#0f766e")
axes[1].set_xlabel("DryStretch (km)")
axes[1].set_ylabel("DayHikes")
fig.suptitle("Water facts in the region")
fig.tight_layout()
show_fig(fig)

# One named DayHike with its WaterSources: all of them nearby in grey, the
# Corridor ones in blue, the line in orange.
hikes = dayhikes.set_index("id")
watered = water[(hikes["name_origin"] == "osm_name") & (water["water_count"] >= 1)]
example_id = (
    hikes.index[hikes["name"] == "Kanjon Velika Paklenica"][0]
    if (hikes["name"] == "Kanjon Velika Paklenica").any()
    else watered["water_count"].idxmax()
)
line = hikes.loc[[example_id]]
chain = vertices[vertices["dayhike_id"] == example_id]
spots = pd.concat(
    corridor_amenities(chain, amenities, kind=kind, radius_m=CORRIDOR_M) for kind in WATER_SOURCE_KINDS
)
print(f"example_dayhike\t{example_id}\t{hikes.at[example_id, 'name']}")
print(water.loc[[example_id]].round(1).to_string())
box = line.total_bounds
pad = 0.01
near = amenities[
    amenities["kind"].isin(WATER_SOURCE_KINDS)
    & amenities["longitude"].between(box[0] - pad, box[2] + pad)
    & amenities["latitude"].between(box[1] - pad, box[3] + pad)
]
fig, ax = plt.subplots(figsize=(6, 6))
line.to_crs(epsg=3857).plot(ax=ax, color="#c2410c", linewidth=2)
near[~near["osm_key"].isin(spots["osm_key"])].to_crs(epsg=3857).plot(ax=ax, color="#a8a29e", markersize=25)
near[near["osm_key"].isin(spots["osm_key"])].to_crs(epsg=3857).plot(ax=ax, color="#1d4ed8", markersize=45)
try:
    import contextily as cx

    cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
except Exception as error:  # offline: the line alone is still the map
    print("basemap skipped", type(error).__name__)
ax.set_title(
    f"{hikes.at[example_id, 'name']}: {int(water.at[example_id, 'water_count'])} WaterSources in the Corridor "
    f"(blue), DryStretch {water.at[example_id, 'dry_stretch_m'] / 1000:.1f} km",
    fontsize=9,
)
ax.set_axis_off()
fig.tight_layout()
show_fig(fig)

## Facilities

What the DayHike offers along the way and where it starts. Five yes/no flags
per DayHike:

- Along the DayHike, in the 100 m Corridor: `hut_along` (a Hut),
  `restaurant_cafe_along` (a restaurant or cafe), `shelter_along` (a Shelter).
- At the Trailhead, within 500 m of the Catalog Trailhead:
  `restaurant_cafe_trailhead`, `parking_trailhead`.

**Rules** (`facilities_facts`):

1. Along is the Corridor of the Landscape section: `corridor_amenities` finds
   the amenity points within 100 m of the line (distance to the line, not only
   to the vertices), here for the kinds `hut`, `restaurant_cafe` and `shelter`.
   Reversing the walk changes nothing.
2. At the Trailhead is a plain distance: the point is within 500 m (haversine)
   of the Catalog Trailhead (`trailhead_latitude` and `trailhead_longitude`),
   which is where the main walk starts, near parking when any lies within
   5 km. 450 m counts, 550 m does not. The production
   AmenityFacts count around the Trailhead in a 20 km disk; that disk is not
   used here, so a cafe in the next village does not make the Trailhead a
   meal stop.
3. A Shelter (`amenity=shelter`, a lean-to or weather shelter) is a separate
   kind and never sets the Hut flag, and a Hut never sets the Shelter flag.
4. A flag is plain false when the amenity is not there. Two hikes that both lack
   a Hut agree on that point, and the 05a measure counts that as agreement. The
   flags have no missing state: a DayHike that is not in the data has no row at
   all.

**Data notes.** The restaurant or cafe flags only know what OSM maps, which is
thin in the mountains and thicker in towns and vineyards. The Trailhead is
chosen near parking when any lies within 5 km, so `parking_trailhead` is true more often
than a random point on the map would give. The code below prints how often each
flag is set, how the number of set flags is spread, and how the flags change
with the highest point.

**Region result** (4,584 DayHikes, no flag is ever missing): a Hut along 22.8%,
a restaurant or cafe along 33.9%, a Shelter along 17.0%, a restaurant or cafe at
the Trailhead 37.3%, parking at the Trailhead 59.3%. 24.5% of the DayHikes set
no flag, 24.7% one, 18.7% two, 21.2% three, 9.9% four and 0.9% all five. The
flags follow the terrain the way one would expect: a Hut is along 4% of the
DayHikes that stay under 500 m, 25% of those reaching 500 to 1,500 m and 41% of
those above 1,500 m, while a restaurant or cafe along the way falls from 44% to
33% to 23% and parking at the Trailhead from 65% to 63% to 45%.

In [ ]:
facilities = facilities_facts(dayhikes, vertices, amenities)
print("facilities missing share")
print(facilities.isna().mean().round(3).to_string())
for flag in FACILITIES_FLAG_COLUMNS:
    print(f"{flag}\t{int(facilities[flag].sum())}\tof\t{len(facilities)}\t{facilities[flag].mean():.3f}")
flag_count = facilities.sum(axis=1)
print("flags_set_share\t" + str(flag_count.value_counts(normalize=True).sort_index().round(3).to_dict()))

bands = pd.cut(
    altitude["highest_m"], [-1, 500, 1500, 10_000], labels=["under 500 m", "500-1500 m", "over 1500 m"]
)
print("share of DayHikes with each flag, by highest point")
print(facilities.groupby(bands, observed=True).mean().round(3).T.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
shares = facilities.mean()
axes[0].barh([c.replace("_", " ") for c in shares.index], shares.to_numpy(), color="#7c3aed")
axes[0].invert_yaxis()
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("share of region DayHikes")
counts = flag_count.value_counts().reindex(range(6), fill_value=0)
axes[1].bar(counts.index.astype(str), counts.to_numpy(), color="#0f766e")
axes[1].set_xlabel("flags set (of 5)")
axes[1].set_ylabel("DayHikes")
fig.suptitle("Facilities facts in the region")
fig.tight_layout()
show_fig(fig)

# One named DayHike: the line in orange, the Trailhead and its 500 m disc in
# black, the counted amenities in colour (grey: nearby but not counted).
hikes = dayhikes.set_index("id")
example_name = "Šmartno - Koča pod Kremžarjevim vrhom"
named = hikes[hikes["name_origin"] == "osm_name"]
example_id = (
    hikes.index[hikes["name"] == example_name][0]
    if (hikes["name"] == example_name).any()
    else flag_count.loc[named.index].idxmax()
)
line = hikes.loc[[example_id]]
print(f"example_dayhike\t{example_id}\t{hikes.at[example_id, 'name']}")
print(facilities.loc[[example_id]].to_string())
trailhead = gpd.GeoSeries.from_xy(
    line["trailhead_longitude"], line["trailhead_latitude"], crs="EPSG:4326"
)
metric = line.estimate_utm_crs()
disc = trailhead.to_crs(metric).buffer(TRAILHEAD_M).to_crs(epsg=3857)
box = line.total_bounds
pad = 0.01
kinds = ("hut", "restaurant_cafe", "shelter", "parking")
near = amenities[
    amenities["kind"].isin(kinds)
    & amenities["longitude"].between(box[0] - pad, box[2] + pad)
    & amenities["latitude"].between(box[1] - pad, box[3] + pad)
]
chain = vertices[vertices["dayhike_id"] == example_id]
along_keys = {
    kind: set(corridor_amenities(chain, amenities, kind=kind, radius_m=CORRIDOR_M)["osm_key"])
    for kind in ("hut", "restaurant_cafe", "shelter")
}
from_trailhead = near.to_crs(metric).distance(trailhead.to_crs(metric).iloc[0])
counted = near.apply(
    lambda row: row["osm_key"] in along_keys.get(row["kind"], ())
    or (row["kind"] in ("restaurant_cafe", "parking") and from_trailhead[row.name] <= TRAILHEAD_M),
    axis=1,
)
colors = {"hut": "#b91c1c", "restaurant_cafe": "#1d4ed8", "shelter": "#15803d", "parking": "#7c3aed"}
fig, ax = plt.subplots(figsize=(6, 6))
line.to_crs(epsg=3857).plot(ax=ax, color="#c2410c", linewidth=2)
disc.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.2)
if (~counted).any():
    near[~counted].to_crs(epsg=3857).plot(ax=ax, color="#a8a29e", markersize=20)
for kind, color in colors.items():
    chosen = near[counted & (near["kind"] == kind)]
    if len(chosen):
        chosen.to_crs(epsg=3857).plot(ax=ax, color=color, markersize=55, label=kind.replace("_", "/"))
try:
    import contextily as cx

    cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
except Exception as error:  # offline: the line alone is still the map
    print("basemap skipped", type(error).__name__)
ax.legend(fontsize=8, loc="best")
ax.set_title(
    f"{hikes.at[example_id, 'name']}: {int(flag_count[example_id])} of 5 flags set (circle: 500 m round the Trailhead)",
    fontsize=9,
)
ax.set_axis_off()
fig.tight_layout()
show_fig(fig)

## Trail

What the DayHike walks on, apart from where it is. Five facts per DayHike,
from its walked member ways (`in_chain = 1`; Leftovers are not walked):

- Four SurfaceMix shares (`paved_share`, `gravel_share`, `natural_share`,
  `rock_share`): the length fraction of each class, length being the metres of
  the way on the walk (`walked_m`). They sum to 1 over the ways that could be classified.
- `tarmac_approach_m`: TarmacApproach. Start at the Catalog Trailhead, where
  the walk starts, and add up the walked length of consecutive paved ways until
  the first way that is not paved. It is 0 when the first way is unpaved, and the
  whole classified length when every way is paved. The ways are taken in the
  stored walk order (`walk_order`, not member order), and a loop counts the
  shorter of its two directions, so the walking direction does not matter.

**Classifying a way.** The `surface` tag decides when it is one of these
(first `;` token, lowercased):

| Class | `surface` values |
| --- | --- |
| paved | asphalt, concrete (and `concrete:*`), paving_stones, sett, cobblestone, paved |
| gravel | compacted, gravel, fine_gravel, unpaved, pebblestone |
| natural | ground, dirt, earth, grass, mud, sand, woodchips |
| rock | rock, bare_rock, stone |

A way with no `surface`, or an unlisted one (`wood`, `metal`, ...), is
classified from its `highway`: roads (motorway down to residential, service,
living_street, pedestrian, cycleway, unclassified, and their `_link` forms)
are paved; a `track` follows its `tracktype` (grade1 paved, grade2-3 gravel,
grade4-5 natural, none gravel); `path`, `footway`, `steps` and `bridleway` are
natural; `via_ferrata` is rock. Any other way is left out of the denominator
(and ends the TarmacApproach). A DayHike with no classified way has all five
facts missing, not 0.

In [ ]:
trail = trail_facts(dayhikes, vertices, ways)
print(trail.describe().round(3).to_string())
print("trail missing share")
print(trail.isna().mean().round(3).to_string())

walked = ways[ways["in_chain"] == 1].drop_duplicates("way_id")
bags = [json.loads(tags) for tags in walked["tags"]]
by_tag = sum(surface_class_from_tag(bag) is not None for bag in bags)
by_fallback = sum(
    surface_class_from_tag(bag) is None and surface_class(bag) is not None for bag in bags
)
print(
    f"ways_classified\tby_tag={by_tag}\tby_highway_fallback={by_fallback}\t"
    f"left_out={len(bags) - by_tag - by_fallback}\tof\t{len(bags)}"
)
print(
    "fallback_highways",
    Counter(
        bag.get("highway")
        for bag in bags
        if surface_class_from_tag(bag) is None and surface_class(bag) is not None
    ).most_common(8),
)
print(f"tarmac_zero_share\t{(trail['tarmac_approach_m'] == 0).mean():.3f}")

SURFACE_COLORS = {"paved": "#475569", "gravel": "#d97706", "natural": "#15803d", "rock": "#a8a29e"}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
mean_mix = trail[list(SURFACE_SHARE_COLUMNS)].mean()
left = 0.0
for kind, column in zip(SURFACE_COLORS, SURFACE_SHARE_COLUMNS):
    axes[0].barh([0], [mean_mix[column]], left=left, color=SURFACE_COLORS[kind], edgecolor="white", label=kind)
    left += mean_mix[column]
axes[0].set_xlim(0, 1)
axes[0].set_yticks([])
axes[0].set_xlabel("mean share of the DayHike")
axes[0].set_title("SurfaceMix (four shares)", fontsize=10)
axes[0].legend(fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.3))
axes[1].hist(trail["paved_share"].dropna(), bins=20, color=SURFACE_COLORS["paved"])
axes[1].set_xlabel("paved share")
axes[1].set_ylabel("DayHikes")
axes[2].hist(trail["tarmac_approach_m"].dropna().clip(upper=6_000) / 1_000, bins=30, color="#7c3aed")
axes[2].set_xlabel("TarmacApproach (km, clipped at 6)")
axes[2].set_ylabel("DayHikes")
fig.suptitle("Trail facts in the region")
fig.tight_layout()
show_fig(fig)

# One DayHike with a road approach and three or more surfaces, ways coloured by class.
classes_present = (trail[list(SURFACE_SHARE_COLUMNS)] > 0.05).sum(axis=1)
candidates = trail[(trail["tarmac_approach_m"] > 0) & (classes_present >= 3)]
example_id = int((candidates if len(candidates) else trail.dropna()).index[0])
example = ways[(ways["dayhike_id"] == example_id) & (ways["in_chain"] == 1)].copy()
example["surface_class"] = [surface_class(json.loads(tags)) or "unknown" for tags in example["tags"]]
example = example[example.geometry.notna()]
print(f"example_dayhike\t{example_id}\t{dayhikes.set_index('id').at[example_id, 'name']}")
print(trail.loc[[example_id]].round(3).to_string())
fig, ax = plt.subplots(figsize=(6, 5))
for kind, color in {**SURFACE_COLORS, "unknown": "#dc2626"}.items():
    part = example[example["surface_class"] == kind]
    if len(part):
        part.plot(ax=ax, color=color, linewidth=3, label=kind)
ax.legend(fontsize=8)
ax.set_title(f"Ways of DayHike {example_id} by surface class", fontsize=10)
fig.tight_layout()
show_fig(fig)

## RoundTrip

Whether the DayHike's length is the whole walk back to the car. One fact per
DayHike:

- `round_trip`: True when the last vertex of the main walk lies within 50 m of
  the first, the Trailhead. 50 m is `GAP_JOIN_M`, the gap at which two member
  ways count as connected.

**Rule** (`round_trip_facts`):

1. The haversine distance between the first and the last vertex of
   DayHikeGeometry. Swapping the ends leaves it alone, so the walking direction
   does not matter.
2. 50 m or less is a RoundTrip: walking `length_m` brings the Hiker back to the
   Trailhead.
3. A lollipop is not a RoundTrip. The main walk of a loop with a stem
   (ADR-0012) starts at the stem's far end, walks the stem and the loop, and
   stops where the stem meets the loop. To get back the Hiker walks the stem
   again, which `length_m` does not hold.
4. A line is not a RoundTrip. OSM does not say whether it is walked
   point-to-point (a shuttle or a bus back) or out-and-back (about twice
   `length_m`); its length is one way either way.
5. The fact is never missing: every DayHike has a main walk.

The Trail Facet's TarmacApproach uses the same 50 m test to decide when the
walk may start from either end.

The code below prints how many region DayHikes are RoundTrips, lollipops and
lines (a lollipop here: the walk ends within 50 m of its own line, leaving out
the last 200 m), draws the distance between the ends on a log scale, and maps
one of each kind with its first vertex (green) and last vertex (red).

**Region result** (4,584 DayHikes, never missing): 386 RoundTrips (8.4%), 186
lollipops (4.1%) and 4,012 lines (87.5%). The 50 m cut sits in a gap: 366
DayHikes end within 10 m of where they start, only 20 more within 50 m, 30 more
within 100 m. A line's ends are a median 67% of its length apart, so most lines
are point-to-point or out-and-back, not near misses. For nine in ten region
DayHikes `length_m` is therefore not the whole walk back to the car
(`docs/future-work.md`, "Walking back on a line").

In [ ]:
round_trip = round_trip_facts(dayhikes, vertices)
print("round_trip missing share")
print(round_trip.isna().mean().round(3).to_string())

hikes = dayhikes.set_index("id")
lines = hikes.geometry.to_crs(dayhikes.estimate_utm_crs())


def end_gap_m(line):
    return Point(line.coords[0]).distance(Point(line.coords[-1]))


def ends_on_own_walk(line):
    return line.length > 400 and Point(line.coords[-1]).distance(substring(line, 0, line.length - 200)) <= GAP_JOIN_M


end_gap = lines.map(end_gap_m)
kind = pd.Series("line", index=hikes.index)
kind[lines.map(ends_on_own_walk)] = "lollipop"
kind[round_trip["round_trip"]] = "round_trip"
for name in ("round_trip", "lollipop", "line"):
    print(f"walk_kind\t{name}\t{int((kind == name).sum())}\tof\t{len(kind)}\t{(kind == name).mean():.3f}")
for metres in (10, 50, 100, 500):
    print(f"ends_within\t{metres} m\t{int((end_gap <= metres).sum())}")
gap_share = (end_gap / hikes["length_m"])[kind == "line"]
print(f"line_end_gap_share_of_length\tmedian={gap_share.median():.2f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(np.log10(end_gap.clip(lower=1)), bins=40, color="#0f766e")
ax.axvline(np.log10(GAP_JOIN_M), color="#c2410c", linestyle="--", label="50 m")
ax.set_xticks([0, 1, 2, 3, 4], ["1 m", "10 m", "100 m", "1 km", "10 km"])
ax.set_xlabel("distance between first and last vertex (log)")
ax.set_ylabel("DayHikes")
ax.set_title("RoundTrip facts in the region")
ax.legend()
fig.tight_layout()
show_fig(fig)

# One named DayHike of each kind: the line, its first vertex (green, the
# Trailhead) and its last vertex (red).
EXAMPLES = {
    "round_trip": "Po stopinjah pastirjev",
    "lollipop": "Arriacher Hofwanderweg-Variante",
    "line": "Velika planina - Planina Dol",
}
named = hikes["name_origin"] == "osm_name"
examples = {}
for name, example_name in EXAMPLES.items():
    if (hikes["name"] == example_name).sum() == 1:
        examples[name] = hikes.index[hikes["name"] == example_name][0]
    elif ((kind == name) & named).any():
        examples[name] = hikes.index[(kind == name) & named][0]
fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4), squeeze=False)
for ax, (name, example_id) in zip(axes[0], examples.items()):
    print(f"example_dayhike\t{name}\t{example_id}\t{hikes.at[example_id, 'name']}\tend_gap_m={end_gap[example_id]:.0f}")
    one = hikes.loc[[example_id]].to_crs(epsg=3857)
    one.plot(ax=ax, color="#c2410c", linewidth=2)
    coords = one.geometry.iloc[0].coords
    ax.scatter(*coords[0], s=60, color="#15803d", zorder=5)
    ax.scatter(*coords[-1], s=30, color="#b91c1c", zorder=6)
    try:
        import contextily as cx

        cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
    except Exception as error:  # offline: the line alone is still the map
        print("basemap skipped", type(error).__name__)
    ax.set_title(f"{name}: {hikes.at[example_id, 'name']}\nends {end_gap[example_id]:.0f} m apart", fontsize=8)
    ax.set_axis_off()
fig.tight_layout()
show_fig(fig)

## Steepness

How steep the climbing is, apart from how long the DayHike is and how much it
climbs in all. Effort holds the totals; Steepness lets the Hiker ask for "as
steep as this one" at any length. Two facts per DayHike:

- `climb_grade` (ClimbGrade): the elevation gain over the length of the
  stretches that climb, a fraction (0.12 is 12 %).
- `steepest_km_m` (SteepestKm): the largest height change, up or down, over any
  1 km of the walk, in metres. The hardest sustained stretch of the day.

**Rule** (`steepness_facts`):

1. Elevation is resampled every ~50 m (`STEEPNESS_STEP_M`) of haversine
   distance along the main walk, interpolated between the vertices that have
   `ele`. Mappers place vertices 5 m or 50 m apart; on raw vertices the DEM's
   metre-scale jitter between close vertices adds climb, so resampling keeps
   both facts the same for any vertex spacing.
2. ClimbGrade takes the steps that rise in the direction with the larger climb
   (the same choice as Effort's `elevation_gain_m`) and divides their rise by
   their length. Flat and falling stretches do not dilute it, so a RoundTrip,
   which walks back down, and a line up the same slope score alike. Plain
   gain / length would halve the RoundTrip: on a walk that goes up and
   straight back down it is half the ClimbGrade.
3. SteepestKm is the largest |height change| between two resampled points 1 km
   (`STEEPEST_KM_WINDOW_M`) apart. Taking the absolute value keeps it the same
   in both walking directions. A walk shorter than 1 km uses its two ends.
4. Fewer than two vertices with `ele` leaves both missing; a flat walk is 0,
   not missing.

The share of length steeper than 30 % was also tried: it ranks the region
almost the same as ClimbGrade (Spearman 0.88), so it is not a third fact.

The code below prints the spread, ClimbGrade against plain gain / length
(Effort's `elevation_gain_m` over `length_m`) by RoundTrip, and the named
DayHikes at the 5th, 50th and 95th ClimbGrade percentile.

**Region result** (4,584 DayHikes, never missing): ClimbGrade has a median of
12 % (middle half 8–17 %, 95th percentile 27 %) and SteepestKm a median of
149 m (middle half 91–224 m, 95th percentile 349 m). Plain gain / length is a
median 0.66 of ClimbGrade on RoundTrips against 0.89 on the other DayHikes, so
it would rank a RoundTrip as gentler than a line up the same slope. (Both
ratios sit above the ideal 0.5 and 1 because Effort's raw-vertex gain carries
DEM jitter, `docs/future-work.md`, "Smoothed elevation gain".) The steepest
DayHikes are via ferratas and summit paths, with ClimbGrade up to 79 %.

In [ ]:
steepness = steepness_facts(dayhikes, vertices)
print(steepness.describe().round(3).to_string())
print("steepness missing share")
print(steepness.isna().mean().round(3).to_string())
print(f"steepness_step_m\t{STEEPNESS_STEP_M}\twindow_m\t{STEEPEST_KM_WINDOW_M}")

plain_grade = effort["elevation_gain_m"] / effort["length_m"]
by_kind = pd.DataFrame(
    {
        "climb_grade": steepness["climb_grade"],
        "plain_grade": plain_grade,
        "round_trip": round_trip["round_trip"],
    }
).dropna()
for is_round, group in by_kind.groupby("round_trip"):
    print(
        f"plain_over_climb_grade\tround_trip={is_round}\t{len(group)}"
        f"\tmedian={(group['plain_grade'] / group['climb_grade'].clip(lower=1e-9)).median():.2f}"
    )
named_grades = steepness["climb_grade"][hikes["name_origin"] == "osm_name"].dropna()
for quantile in (0.05, 0.5, 0.95):
    example_id = (named_grades - named_grades.quantile(quantile)).abs().idxmin()
    print(
        f"steepness_example\tp{round(quantile * 100)}\t{hikes.at[example_id, 'name']}"
        f"\tclimb_grade={steepness.at[example_id, 'climb_grade']:.3f}"
        f"\tsteepest_km_m={steepness.at[example_id, 'steepest_km_m']:.0f}"
    )

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(steepness["climb_grade"].dropna() * 100, bins=40, color="#0f766e")
axes[0].set_xlabel("ClimbGrade (%)")
axes[0].set_ylabel("DayHikes")
axes[1].hist(steepness["steepest_km_m"].dropna(), bins=40, color="#c2410c")
axes[1].set_xlabel("SteepestKm (m over 1 km)")
axes[1].set_ylabel("DayHikes")
for is_round, color in ((False, "#a8a29e"), (True, "#7c3aed")):
    group = by_kind[by_kind["round_trip"] == is_round]
    axes[2].scatter(
        group["climb_grade"] * 100, group["plain_grade"] * 100, s=4, color=color,
        label="RoundTrip" if is_round else "not a RoundTrip",
    )
axes[2].plot([0, 60], [0, 60], color="#57534e", linewidth=0.8, linestyle="--")
axes[2].set_xlim(0, 60)
axes[2].set_ylim(0, 60)
axes[2].set_xlabel("ClimbGrade (%)")
axes[2].set_ylabel("gain / length (%)")
axes[2].legend(fontsize=8)
fig.suptitle("Steepness facts in the region")
fig.tight_layout()
show_fig(fig)

## Similarity facts table

One row per region DayHike, indexed by DayHike id: `name`, `name_origin`, the
Trailhead and the DayHike line (for maps in `05a`), then every Facet's facts.
It is written as a pandas pickle next to the Catalog,
`{catalog}.similarity_facts.pkl` (`data/catalog.similarity_facts.pkl` for the
local Catalog). It is regenerable and gitignored (`*.similarity_facts.pkl`).

In [ ]:
facts = gpd.GeoDataFrame(
    dayhikes.set_index("id")[
        ["name", "name_origin", "trailhead_latitude", "trailhead_longitude", "geometry"]
    ]
    .join(effort)
    .join(altitude)
    .join(profile)
    .join(landscape)
    .join(weather)
    .join(water)
    .join(facilities)
    .join(trail)
    .join(round_trip)
    .join(steepness),
    geometry="geometry",
    crs=dayhikes.crs,
)
facts_path = similarity_facts_path_for(catalog_path)
facts.to_pickle(facts_path)
print("similarity_facts", facts_path)
print(f"similarity_facts_rows\t{len(facts)}")
print("columns", list(facts.columns))